
<img src="https://reqlut2.s3.sa-east-1.amazonaws.com/reqlut-images/duoc/logo.png?v=87.8" width="180px"/>


In [ ]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

# Hiperparámetros modelo Random Forest

In [2]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import randint
import pandas as pd
import numpy as np
#from src.carga_csv import cargar_csv
import json

## Predicción de la depresión en estudiantes

Este cuaderno muestra un flujo de trabajo de aprendizaje automático para predecir la depresión en estudiantes basándose en diversas características. El proceso incluye la carga de datos, la exploración inicial, la división de datos, el entrenamiento del modelo de referencia, el ajuste de hiperparámetros mediante `GridSearchCV` y `RandomizedSearchCV`, y la evaluación del modelo final.

### 1. Carga de datos y exploración inicial

En primer lugar, cargamos el conjunto de datos preprocesado sobre la depresión en estudiantes y realizamos una inspección inicial de su estructura y contenido.


In [3]:
df = pd.read_csv("Student_Depression_Dataset_Codificado.csv")

In [ ]:
df = cargar_csv(
    r"..\data\processed\Student_Depression_Dataset_codificado.csv"
    )
df.head()

,cat__Gender_Male,cat__Have you ever had suicidal thoughts ?_Yes,cat__Family History of Mental Illness_Yes,bin__Degree_0,bin__Degree_1,bin__Degree_2,bin__Degree_3,bin__Degree_4,ord__Academic Pressure,ord__Study Satisfaction,ord__Financial Stress,ord__Sleep Duration,ord__Dietary Habits,num__Age,num__CGPA,num__Work/Study Hours,Depression
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.345273,-0.694638,-1.489063,-0.354328,1.374420,1.476593,0.895400,-1.122130,1
1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.827109,1.510988,-0.793223,-0.354328,0.119629,-0.370683,-1.200717,-1.122130,0
2,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,-0.102982,1.510988,-1.489063,-1.241729,1.374420,1.066087,-0.429182,0.496561,0
3,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,-0.102982,-0.694638,1.294297,0.533072,0.119629,0.450328,-1.412377,-0.852348,1
4,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.621145,0.040570,-1.489063,-0.354328,0.119629,-0.165430,0.321870,-1.661693,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27817 entries, 0 to 27816
Data columns (total 17 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   cat__Gender_Male                                27817 non-null  float64
 1   cat__Have you ever had suicidal thoughts ?_Yes  27817 non-null  float64
 2   cat__Family History of Mental Illness_Yes       27817 non-null  float64
 3   bin__Degree_0                                   27817 non-null  float64
 4   bin__Degree_1                                   27817 non-null  float64
 5   bin__Degree_2                                   27817 non-null  float64
 6   bin__Degree_3                                   27817 non-null  float64
 7   bin__Degree_4                                   27817 non-null  float64
 8   ord__Academic Pressure                          27817 non-null  float64
 9   ord__Study Satisfaction                

La salida de `df.head()` muestra las primeras 5 filas del conjunto de datos, que contiene varias características categóricas (codificadas one-hot), binarias y numéricas. La variable objetivo, 'remainder__Depression', indica la presencia (1.0) o ausencia (0.0) de depresión.

`df.info()` confirma que hay 27,817 entradas y 17 columnas. No tienen valores faltantes, lo que indica que los datos ya han sido preprocesados y limpiados. Esto es crucial para proceder con el entrenamiento del modelo sin pasos adicionales de imputación.

In [5]:
# Cargar datasets
#train_df = pd.read_csv('../data/processed/Student_Depression_Dataset_Train.csv')
#test_df = pd.read_csv('../data/processed/Student_Depression_Dataset_Test.csv')

train_df = pd.read_csv("Student_Depression_Dataset_Train.csv")
test_df = pd.read_csv("Student_Depression_Dataset_Test.csv")

# Conjunto de entrenamiento
X_train = train_df.drop(columns=['Depression'])
y_train = train_df['Depression']

# Conjunto de prueba
X_test = test_df.drop(columns=['Depression'])
y_test = test_df['Depression']

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de y_train:", y_train.shape)

print("Dimensiones de X_test:", X_test.shape)
print("Dimensiones de y_test:", y_test.shape)

Dimensiones de X_train: (22253, 16)
Dimensiones de y_train: (22253,)
Dimensiones de X_test: (5564, 16)
Dimensiones de y_test: (5564,)


### 2. Preparación de Datos

Antes de entrenar cualquier modelo, el conjunto de datos se divide en características (X) y la variable objetivo (y). Posteriormente, estos se dividen en conjuntos de entrenamiento y prueba para evaluar el rendimiento del modelo con datos no vistos. `test_size=0.2` indica que el 20% de los datos se utilizará para la prueba, y `random_state=42` asegura la reproducibilidad de la división.

In [6]:
print("X_train head:")
display(X_train.head())

X_train head:


,cat__Gender_Male,cat__Have you ever had suicidal thoughts ?_Yes,cat__Family History of Mental Illness_Yes,bin__Degree_0,bin__Degree_1,bin__Degree_2,bin__Degree_3,bin__Degree_4,ord__Academic Pressure,ord__Study Satisfaction,ord__Financial Stress,ord__Sleep Duration,ord__Dietary Habits,num__Age,num__CGPA,num__Work/Study Hours
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.342450,-0.692243,-0.090078,-1.239966,0.121880,0.242172,-0.358151,0.496245
1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.618804,1.514176,-0.785601,-0.353132,-1.133187,0.242172,0.664384,-0.043712
2,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.342450,-1.427716,1.300969,-1.239966,1.376948,-0.167751,1.080214,0.496245
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.104841,-0.692243,-1.481125,-0.353132,1.376948,-1.602482,1.516496,-0.853649
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,-0.104841,0.778703,0.605446,-0.353132,0.121880,0.242172,-1.435221,-0.313691


In [7]:
print("y_train head:")
display(y_train.head())

y_train head:


,Depression
0,1
1,1
2,1
3,0
4,1


Las salidas de `X_train.head()` y `y_train.head()` ofrecen un vistazo a los datos de entrenamiento después de la división. `X_train` contiene el conjunto de características, mientras que `y_train` contiene las etiquetas objetivo correspondientes. Esta verificación asegura que los datos han sido preparados correctamente para el entrenamiento del modelo.

In [8]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy del modelo RandomForestClassifier: {accuracy:.4f}")

Accuracy del modelo RandomForestClassifier: 0.8408


### 3. Entrenamiento del Modelo Base: RandomForestClassifier

Se entrena un `RandomForestClassifier` con hiperparámetros predeterminados como modelo base. La precisión del modelo en el conjunto de prueba se calcula para establecer un punto de referencia para la comparación con modelos con hiperparámetros ajustados.

### Búsqueda de hiperparámetros con GridSearchCV para RandomForestClassifier

Ahora, utilizaremos `GridSearchCV` para una búsqueda exhaustiva de hiperparámetros en el modelo `RandomForestClassifier`. Esto nos permitirá encontrar la combinación óptima para mejorar su rendimiento.

In [9]:
# # ES DEMASIADO COSTOSO COMPUTACIONALMENTE, NO EJECUTA.

# # Definir el modelo RandomForestClassifier
# rf_gs = RandomForestClassifier(random_state=42)

# # Definir la cuadrícula de parámetros para GridSearchCV (ajustada para RandomForestClassifier)
# param_grid_rf_gs = {
#     'n_estimators': [100, 200, 300],  # Número de árboles en el bosque
#     'max_features': ['sqrt', 'log2'], # Número de características a considerar en cada división
#     'max_depth': [10, 20, None],      # Profundidad máxima del árbol (None significa sin límite)
#     'min_samples_split': [2, 5],      # Número mínimo de muestras requeridas para dividir un nodo interno
#     'min_samples_leaf': [1, 2]        # Número mínimo de muestras requeridas en cada nodo hoja
# }

# # Inicializar GridSearchCV para RandomForestClassifier
# grid_search_rf = GridSearchCV(
#     estimator=rf_gs,
#     param_grid=param_grid_rf_gs,
#     cv=5,       # Validación cruzada con 5 folds
#     n_jobs=-1,  # Usar todos los núcleos disponibles
#     verbose=2   # Mostrar detalles del progreso
# )

# # Ajustar GridSearchCV a los datos de entrenamiento
# grid_search_rf.fit(X_train, y_train)

# # Mostrar los mejores parámetros y la mejor puntuación
# print("\nMejores parámetros para RandomForestClassifier (GridSearchCV):", grid_search_rf.best_params_)
# print("Mejor puntuación (accuracy) para RandomForestClassifier (GridSearchCV):", grid_search_rf.best_score_)

# # Evaluar el modelo con los mejores hiperparámetros en el conjunto de prueba
# best_rf_model_gs = grid_search_rf.best_estimator_
# y_pred_rf_gs = best_rf_model_gs.predict(X_test)
# accuracy_rf_gs = accuracy_score(y_test, y_pred_rf_gs)
# print("Precisión de RandomForestClassifier (GridSearchCV) en el conjunto de prueba:", accuracy_rf_gs)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


KeyboardInterrupt: 

### 4. Ajuste de Hiperparámetros con GridSearchCV

`GridSearchCV` realiza una búsqueda exhaustiva sobre una cuadrícula de parámetros especificada para encontrar la mejor combinación de hiperparámetros para el `RandomForestClassifier`. Aunque es eficaz, este método puede ser computacionalmente costoso, especialmente con un espacio de parámetros y un conjunto de datos grandes. El comentario en el código refleja esto, indicando que fue demasiado costoso ejecutarlo completamente en este entorno.

### Búsqueda de hiperparámetros con RandomizedSearchCV para RandomForestClassifier

Dado que `GridSearchCV` puede ser computacionalmente costoso, utilizaremos `RandomizedSearchCV` para explorar un subconjunto aleatorio del espacio de hiperparámetros, lo que a menudo puede encontrar buenos resultados en menos tiempo.

In [17]:
# Modelo base
model_rs = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

# Espacio de búsqueda reducido
param_dist_rs = {
    'n_estimators': randint(100, 300),
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy']
}

# Búsqueda aleatoria ligera
random_search = RandomizedSearchCV(
    estimator=model_rs,
    param_distributions=param_dist_rs,
    n_iter=20,      # antes 100
    cv=3,           # antes 5
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

random_search.fit(X_train, y_train)

print("Mejores parámetros:")
print(random_search.best_params_)

print("Mejor accuracy CV:")
print(f"{random_search.best_score_:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Mejores parámetros:
{'criterion': 'gini', 'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'min_samples_split': 8, 'n_estimators': 120}
Mejor accuracy CV:
0.8433


### 5. Ajuste de Hiperparámetros con RandomizedSearchCV

Para abordar el costo computacional de `GridSearchCV`, se emplea `RandomizedSearchCV`. Este método muestrea un número fijo de configuraciones de parámetros de distribuciones especificadas, ofreciendo una forma más eficiente de explorar el espacio de hiperparámetros. `n_iter` controla el número de combinaciones de parámetros muestreadas, y `cv` especifica el número de divisiones de validación cruzada.

La búsqueda identifica los hiperparámetros óptimos, que luego se utilizan para entrenar un `RandomForestClassifier` refinado.

In [18]:
# Mostrar mejores hiperparámetros encontrados
print(
    "Mejores parámetros de RandomForestClassifier (RandomizedSearchCV):",
    random_search.best_params_
)

# Obtener el mejor modelo encontrado
best_rf_random_model = random_search.best_estimator_

# Predicciones sobre el conjunto de prueba
y_pred_random = best_rf_random_model.predict(X_test)

# Accuracy en test
accuracy_random = accuracy_score(y_test, y_pred_random)

print(
    f"\nPrecisión del modelo final (RandomizedSearchCV) en el conjunto de prueba: "
    f"{accuracy_random:.4f}"
)

# Comparación de rendimiento
print("\n--- Comparación de Rendimiento ---")
print(f"Accuracy por defecto: {accuracy:.4f}")
print(f"Accuracy RandomizedSearchCV: {accuracy_random:.4f}")

if accuracy_random > accuracy:
    print("RandomizedSearchCV obtuvo una mejor precisión.")
elif accuracy > accuracy_random:
    print("El modelo por defecto obtuvo una mejor precisión.")
else:
    print("Ambos métodos obtuvieron la misma precisión.")

Mejores parámetros de RandomForestClassifier (RandomizedSearchCV): {'criterion': 'gini', 'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'min_samples_split': 8, 'n_estimators': 120}

Precisión del modelo final (RandomizedSearchCV) en el conjunto de prueba: 0.8429

--- Comparación de Rendimiento ---
Accuracy por defecto: 0.8408
Accuracy RandomizedSearchCV: 0.8429
RandomizedSearchCV obtuvo una mejor precisión.


### 6. Evaluación y Comparación del Modelo Final

Después de identificar los mejores hiperparámetros utilizando `RandomizedSearchCV`, se entrena un modelo `RandomForestClassifier` final con estas configuraciones optimizadas. La precisión de este modelo se compara con la del modelo base. En este caso, `RandomizedSearchCV` resultó en una precisión ligeramente mejorada, lo que demuestra el beneficio del ajuste de hiperparámetros.

### Cómo usar el hiperparámetro `best_rf_random_model`

Una vez que hemos entrenado `best_rf_random_model` con los mejores hiperparámetros encontrados por `RandomizedSearchCV`, este modelo está listo para realizar predicciones sobre nuevos datos.

**Recuerda:** Los nuevos datos sobre los que quieras predecir deben tener el mismo formato (mismas columnas y preprocesamiento, por ejemplo, escalado si se aplicó) que los datos de entrenamiento (`X_train`).

Aquí te mostramos cómo puedes usarlo para hacer predicciones:

In [1]:
# Acceder a los mejores parámetros
best_hyperparameters = random_search.best_params_

print("\nLos mejores hiperparámetros encontrados por RandomizedSearchCV son:")
for param, value in best_hyperparameters.items():
    print(f"  {param}: {value}")

output_filename = r'..\outputs\params\\best_random_forest_hyperparameters.json'
with open(output_filename, 'w') as f:
    json.dump(best_hyperparameters, f, indent=4)

print(f"\nLos mejores hiperparámetros también se han guardado en '{output_filename}'")

NameError: name 'random_search' is not defined

### 7. Exportación de los Mejores Hiperparámetros

Finalmente, los mejores hiperparámetros encontrados por `RandomizedSearchCV` se extraen y se guardan en un archivo JSON. Esto permite una fácil reproducibilidad y despliegue del modelo optimizado sin necesidad de volver a ejecutar la búsqueda de hiperparámetros cada vez.